# EDA and Submission for Playground Series - Season 6 Episode 4: Predicting Irrigation Need 

## EDA

Load the training data and check for missing values.

In [ ]:
import pandas as pd

training_data = '/kaggle/input/competitions/playground-series-s6e4/train.csv'
df_tv = pd.read_csv(training_data)
print(f'Missing Values:\n{df_tv.isnull().sum()}')

Great, there's no missing values! Now let's check the class balance of our training dataset.

In [ ]:
counts = df_tv['Irrigation_Need'].value_counts()
counts.plot(kind='bar')

There's quite a large imbalance between classes, especially for 'High'. We could split our data into training
and validation sets, then oversample the minority class ('High', and possibly do the same for 'Medium'). That's a huge
discrepancy though, especially for 'High' and I worry about the quality of the oversampled data, even when using something like SMOTE.
Thankfully we can choose classifiers that can handle class imbalance directly, by using weights proportional to the size of the class.

Next let's encode the ordinal class labels, remove the ids, and one-hot encode the
nominal features.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
df_tv['Irrigation_Need'] = ordinal_encoder.fit_transform(df_tv[['Irrigation_Need']])
df_tv.drop(columns=['id'], inplace=True)
df_dummy = pd.get_dummies(df_tv, dtype=int, drop_first=False)

Map the class labels to integers and calculate the correlation matrix.

In [ ]:

corr = df_dummy.corr()
corr.plot(kind='bar', y='Irrigation_Need', figsize=(10,5))

In [ ]:
df_corr_sort = corr.sort_values(by='Irrigation_Need', ascending=False)['Irrigation_Need']
print(df_corr_sort)

We can see that only a few of the features in the training data strongly correlate (positively or negatively) with our class label.
We could use this information to select only the most significant features, say with a correlation threshold > 0.1, for training.
Thankfully the classifiers we will use (gradient boosting trees) tend to handle feature selection innately so we can try training
on all the features as well, and compare the results.

Now let's create our numpy arrays of data and use a label encoder for reproducibility on transforming the test set.

In [ ]:
selected_features = False

if selected_features:
    df_corr_sort_abs = corr.abs().sort_values(by='Irrigation_Need', ascending=False)['Irrigation_Need']
    df_dummy = df_dummy[df_corr_sort_abs[df_corr_sort_abs > 0.1].index]

x = df_dummy.loc[:, df_dummy.columns != 'Irrigation_Need'].values
y = df_dummy.loc[:,'Irrigation_Need'].values

## Training

We will train three different gradient boosting classifier algorithms on our data. These classifiers are LightGBM, CatBoost, and HistGradientBoost.
The following parameters were chosen through hperparameter tuning using Optuna and MLFlow with 5-fold cross validation to choose the best model.
The tuning process is left out for brevity and runtime but the (rough) code can be found on my github for this project.

These three classifiers all have the capability to handle unbalanced data like we have, by weighting the classes according to their size. We make sure each classifier has the appropriate flag to turn this capability on. Also, we can set our objective and scoring functions to consider balanced accuracy, so that we don't overestimate our model's ability in the event it fails to properly label one or more of the classes.

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(class_weight="balanced", n_estimators=5800, 
                    learning_rate=0.48062143716339184, max_depth=2, reg_lambda=65.55085921107225,
                    max_bins=254, reg_alpha=63.39494767057321, min_data_in_leaf=50,
                    boosting_type='gbdt', is_unbalance=True,objective='multiclass', 
                    random_state=1, verbose=-1)

In [ ]:
from catboost import CatBoostClassifier

params = {'iterations': 2900,
          'depth': 3,
          'learning_rate': 0.08449601053998856,
          'loss_function': 'MultiClass',
          'verbose': False,
          'task_type': 'GPU',
          'reg_lambda': 43.682320439250816,
          'bagging_temperature': 0.7971955777573961,
          'auto_class_weights': 'Balanced',
          'random_state': 1}
                        
from sklearn.base import clone

class CustomCatBoostClassifier(CatBoostClassifier):
    def __sklearn_clone__(self):
        return CustomCatBoostClassifier(**self.get_params())
    
cb = CustomCatBoostClassifier(**params)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(class_weight='balanced', max_iter=3300, 
                                    learning_rate=0.42402398491473303, max_depth=2, l2_regularization=12.907091668087006,
                                    loss='log_loss', max_bins=221, random_state=1)

## Ensembling

Now let's ensemble the three models using a simple majority voting classifier.

In [ ]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(estimators=[('hgb', hgb), ('lgbm', lgbm), ('cb', cb)], voting='soft')
voting_clf.fit(x, y)

## Testing and Submission

Now we load the test data, one-hot encode the categorical data, and encode the class labels the same way we did the test data.
Then we will make our predictions and write the submission file.

In [ ]:

import os

testing_data = '/kaggle/input/competitions/playground-series-s6e4/test.csv'

df_test = pd.read_csv(testing_data)

ids = df_test['id'].values

df_test_dummy = pd.get_dummies(df_test.iloc[:,1:], drop_first=False, dtype=int)
df_test_dummy = df_test_dummy[df_dummy.columns]
x_test = df_test_dummy.to_numpy()

out_dir = '/kaggle/working/'
df_submission_stack = pd.DataFrame({'id': ids, 'Irrigation_Need': ordinal_encoder.inverse_transform(voting_clf.predict(x_test).reshape(-1, 1)).flatten()})
df_submission_stack.to_csv(os.path.join(out_dir, 'submission.csv'), index=False)

## Further Experiments
This fairly basic workflow produced a decent public score (0.97031). Longer, more expansive hyperparameter tuning could
be done, as well as feature engineering. There are quite a few low correlating features that could possibly be combined in ways to both
reduce the number of features as well as produce a better model. Training a wider variety of models to ensemble and performing more
advanced stacking of them could also possibly improve the score. One of these models could be a neural network, which I tested but never tuned as well as these gradient boosters. You can find the (rough) NN code on my github for this project.